# flux-portrait — LoRA Training (Colab)

Trains a Flux.1 Dev LoRA on your processed photos from `scripts/prepare_data.py`, using [kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts).

**Before running:**
1. Runtime -> Change runtime type -> GPU (A100 or L4 both work; this notebook auto-detects VRAM and enables low-VRAM flags on smaller GPUs).
2. Locally, zip your processed data: `cd data/processed && zip -r ../../processed.zip .` (from the repo root).
3. Have a HuggingFace token ready with access to `black-forest-labs/FLUX.1-dev` (visit that model page and accept its license first if you haven't).

**What this does:** downloads Flux.1 Dev + text encoders (~28GB total, cached to your Google Drive so this is a one-time cost across sessions), uploads and trains on your photos, and saves the resulting LoRA `.safetensors` back to Drive plus offers a direct download.

**Expect roughly 20-40 minutes of GPU time** for the default 1500 steps, plus first-run download time for the model weights. Run cells top to bottom.

Hyperparameters here mirror `configs/training_config.toml` in the repo — that file is the readable reference copy; this notebook is what actually runs.

In [ ]:
# Cell 1: GPU check + VRAM detection
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if result.returncode != 0 or not result.stdout.strip():
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run."
    )

# Only the first GPU line is used; a multi-GPU runtime would otherwise break
# the 2-value unpack below.
first_gpu_line = result.stdout.strip().splitlines()[0]
gpu_name, mem_str = first_gpu_line.split(", ")
gpu_mem_mb = int(mem_str.strip().split(" ")[0])
LOW_VRAM = gpu_mem_mb < 32000  # True for a 24GB L4, False for a 40GB A100
print(f"Detected GPU: {gpu_name} ({gpu_mem_mb} MiB) -> LOW_VRAM={LOW_VRAM}")

In [ ]:
# Cell 2: Mount Google Drive (persists model cache + LoRA output across
# sessions). This is a convenience, not a hard requirement: Colab's Drive
# auth handshake is known to be flaky ("credential propagation was
# unsuccessful"), especially with strict third-party-cookie blocking or
# certain browsers/extensions. If it fails, fall back to session-local
# storage so training can still proceed -- you'll just re-download models
# next session and rely on the direct-download link at the end instead of
# a Drive copy. If you want Drive persistence working, try: re-run this
# cell (often transient), Runtime -> Restart session, a different browser
# (Chrome tends to be most reliable), or disabling extensions/strict
# cookie blocking for *.google.com.
import os

from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/flux-portrait"
try:
    drive.mount("/content/drive")
    MODEL_CACHE_DIR = f"{DRIVE_ROOT}/models"
    LORA_OUTPUT_DIR = f"{DRIVE_ROOT}/lora"
    DRIVE_MOUNTED = True
except Exception as e:
    print(f"Drive mount failed ({e}); continuing without Drive persistence.")
    MODEL_CACHE_DIR = "/content/models"
    LORA_OUTPUT_DIR = "/content/lora_output"
    DRIVE_MOUNTED = False

os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)
print(f"Model cache: {MODEL_CACHE_DIR}\nLoRA output: {LORA_OUTPUT_DIR}")

In [ ]:
# Cell 3: Clone kohya-ss/sd-scripts and install its dependencies
# (its own requirements.txt already includes bitsandbytes for AdamW8bit)
import os
import subprocess

SD_SCRIPTS_DIR = "/content/sd-scripts"

os.chdir("/content")
if not os.path.exists(SD_SCRIPTS_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/kohya-ss/sd-scripts.git"],
        check=True,
    )

os.chdir(SD_SCRIPTS_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["accelerate", "config", "default"], check=True)
print("sd-scripts ready.")

In [ ]:
# Cell 4: HuggingFace login (needed for the gated FLUX.1-dev files)
import getpass

from huggingface_hub import login

hf_token = getpass.getpass("Enter your HuggingFace token (needs FLUX.1-dev access): ")
login(token=hf_token)

In [ ]:
# Cell 5: Download model weights, cached to Drive so re-running a session is fast
import os
import shutil

from huggingface_hub import hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

MODEL_FILES = [
    ("black-forest-labs/FLUX.1-dev", "flux1-dev.safetensors"),
    ("black-forest-labs/FLUX.1-dev", "ae.safetensors"),
    ("comfyanonymous/flux_text_encoders", "clip_l.safetensors"),
    ("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors"),
]

model_paths = {}
for repo_id, filename in MODEL_FILES:
    cached_path = os.path.join(MODEL_CACHE_DIR, filename)
    if not os.path.exists(cached_path):
        print(f"Downloading {filename} from {repo_id}...")
        try:
            downloaded = hf_hub_download(repo_id=repo_id, filename=filename)
        except HfHubHTTPError as e:
            if e.response is not None and e.response.status_code == 403:
                raise RuntimeError(
                    f"Access denied for {repo_id}. This repo is gated: log into "
                    f"https://huggingface.co/{repo_id} with the SAME account whose "
                    "token you entered above, and accept its license on that page "
                    "(this is separate from just having an account/token). Then "
                    "re-run this cell."
                ) from e
            raise
        # Copy to a temp name and rename atomically, so a session drop or
        # Drive hiccup mid-copy can never leave a truncated file sitting at
        # cached_path (which would otherwise look "already downloaded"
        # forever on future runs).
        tmp_path = cached_path + ".tmp"
        shutil.copy(downloaded, tmp_path)
        os.replace(tmp_path, cached_path)
    else:
        print(f"Using cached {filename}")
    model_paths[filename] = cached_path

FLUX_MODEL = model_paths["flux1-dev.safetensors"]
AE_MODEL = model_paths["ae.safetensors"]
CLIP_L_MODEL = model_paths["clip_l.safetensors"]
T5XXL_MODEL = model_paths["t5xxl_fp8_e4m3fn.safetensors"]
print("All model files ready.")

In [ ]:
# Cell 6: Upload and unzip your processed training photos
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import files

print("Upload processed.zip (zipped contents of your local data/processed/ folder)")
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))

DATA_DIR = "/content/data/processed"
# Clear any leftovers from a previous run in this session so a re-upload
# fully replaces the dataset instead of merging with stale files.
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(DATA_DIR)

# Flatten regardless of how deeply the zip nested things (a plain folder
# compress, a wrapping folder, nested repo-relative paths, ...), and ignore
# macOS zip artifacts: Finder's "Compress" adds a __MACOSX/ metadata folder
# and .DS_Store files alongside the real content, which a naive "unwrap one
# level" wouldn't know to skip.
data_dir_path = Path(DATA_DIR)
for pattern in ("*.png", "*.txt"):
    for path in data_dir_path.rglob(pattern):
        if "__MACOSX" in path.parts:
            continue
        target = data_dir_path / path.name
        if path != target:
            shutil.move(str(path), str(target))

for entry in data_dir_path.iterdir():
    if entry.is_dir():
        shutil.rmtree(entry)

# Check actual image<->caption correspondence by filename, not just counts,
# so a stray or missing caption file doesn't silently slip through.
image_stems = {os.path.splitext(f)[0] for f in os.listdir(DATA_DIR) if f.endswith(".png")}
caption_stems = {os.path.splitext(f)[0] for f in os.listdir(DATA_DIR) if f.endswith(".txt")}
print(f"Extracted {len(image_stems)} images and {len(caption_stems)} captions to {DATA_DIR}")

if not image_stems:
    other_photos = [f for f in os.listdir(DATA_DIR) if f.lower().endswith((".jpg", ".jpeg"))]
    hint = (
        f" Found {len(other_photos)} .jpg/.jpeg file(s) instead — this looks like "
        "data/raw/ (original phone photos) got zipped instead of data/processed/ "
        "(the cropped .png + .txt output of scripts/prepare_data.py). Re-zip with: "
        "cd data/processed && zip -r ../../processed.zip ."
        if other_photos
        else ""
    )
    raise AssertionError(f"No .png images found under {DATA_DIR} after extraction.{hint}")

unmatched = image_stems.symmetric_difference(caption_stems)
assert not unmatched, (
    f"Image/caption mismatch — {len(unmatched)} file(s) without a pair: {sorted(unmatched)[:5]}"
)

In [ ]:
# Cell 7: Write the dataset config (Kohya's own [[datasets]] schema, not a
# flat train_data_dir flag). Mirrors the [dataset] section of
# configs/training_config.toml.
DATASET_CONFIG_PATH = "/content/dataset_config.toml"

dataset_config = f"""
[general]
flip_aug = true
color_aug = false
shuffle_caption = false
caption_extension = ".txt"

[[datasets]]
batch_size = 1
enable_bucket = false
resolution = [1024, 1024]

  [[datasets.subsets]]
  image_dir = "{DATA_DIR}"
  num_repeats = 1
"""

with open(DATASET_CONFIG_PATH, "w") as f:
    f.write(dataset_config)

print(dataset_config)

In [ ]:
# Cell 8: Assemble and run the training command.
# Hyperparameters mirrored from configs/training_config.toml — keep both
# in sync if you change one.
import os
import subprocess

OUTPUT_DIR = "/content/output"
OUTPUT_NAME = "flux_portrait_v1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Remove any pre-existing output from a prior/failed run in this session so
# the save-and-download cell can't be fooled by a stale artifact if this
# training run fails partway through.
stale_output = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}.safetensors")
if os.path.exists(stale_output):
    os.remove(stale_output)

NETWORK_DIM = 16
NETWORK_ALPHA = 16
LEARNING_RATE = 5e-5
LR_SCHEDULER = "cosine_with_restarts"
LR_WARMUP_STEPS = 100
MAX_TRAIN_STEPS = 1500
SAVE_EVERY_N_STEPS = 250
OPTIMIZER_TYPE = "AdamW8bit"

args = [
    "accelerate", "launch", "--num_cpu_threads_per_process", "1",
    f"{SD_SCRIPTS_DIR}/flux_train_network.py",
    f"--pretrained_model_name_or_path={FLUX_MODEL}",
    f"--clip_l={CLIP_L_MODEL}",
    f"--t5xxl={T5XXL_MODEL}",
    f"--ae={AE_MODEL}",
    f"--dataset_config={DATASET_CONFIG_PATH}",
    f"--output_dir={OUTPUT_DIR}",
    f"--output_name={OUTPUT_NAME}",
    "--save_model_as=safetensors",
    "--network_module=networks.lora_flux",
    f"--network_dim={NETWORK_DIM}",
    f"--network_alpha={NETWORK_ALPHA}",
    "--network_train_unet_only",
    f"--learning_rate={LEARNING_RATE}",
    f"--optimizer_type={OPTIMIZER_TYPE}",
    f"--lr_scheduler={LR_SCHEDULER}",
    f"--lr_warmup_steps={LR_WARMUP_STEPS}",
    f"--max_train_steps={MAX_TRAIN_STEPS}",
    f"--save_every_n_steps={SAVE_EVERY_N_STEPS}",
    "--mixed_precision=bf16",
    "--save_precision=bf16",
    "--gradient_checkpointing",
    "--sdpa",
    "--cache_latents",
    "--cache_text_encoder_outputs",
    "--guidance_scale=1.0",
    "--timestep_sampling=flux_shift",
    "--model_prediction_type=raw",
]

if LOW_VRAM:
    args += ["--fp8_base", "--blocks_to_swap=18"]

print("Running:\n  " + " \\\n  ".join(args))

# PYTHONUNBUFFERED=1: without this, the child Python process (accelerate's
# launched worker) detects its stdout isn't a real terminal and switches to
# full block buffering, so tqdm/print output (including step progress) can
# sit invisible for a long time instead of streaming to the cell in real
# time -- looks like a hang even when training is progressing normally.
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
subprocess.run(args, check=True, cwd=SD_SCRIPTS_DIR, env=env)

In [ ]:
# Cell 9: Save the trained LoRA to Drive (if mounted) and offer a direct download
import os
import shutil

from google.colab import files

lora_filename = f"{OUTPUT_NAME}.safetensors"
src = os.path.join(OUTPUT_DIR, lora_filename)
dst = os.path.join(LORA_OUTPUT_DIR, lora_filename)

if not os.path.exists(src):
    raise FileNotFoundError(
        f"Expected output not found at {src} — check the training cell's output for errors."
    )

shutil.copy(src, dst)
if DRIVE_MOUNTED:
    print(f"Saved LoRA to Google Drive: {dst}")
else:
    print(f"Drive wasn't mounted this session, so this is local Colab storage, not Drive: {dst}")
    print(
        "It will be lost when this session ends -- make sure the download below "
        "actually completes, or use the Files panel (refresh it first) to "
        "right-click that path and choose Download, which tends to be more "
        "reliable than the browser download triggered below for larger files."
    )

files.download(src)